# SOILLIQ-based Inundated Area Estimation for Methane Flux

**2026-03-01 — Yu Huang**

Improves on the precipitation-based FSAT estimation by using CLM2 soil liquid water (SOILLIQ) from the TraCE-21ka simulation.  
The degree of saturation of the top soil layers is a physically better proxy for inundation than precipitation, because it integrates precipitation, evapotranspiration, and drainage.

**SOILLIQ approach:**
- Compute depth-weighted degree of saturation for the top 4 soil layers (0–12 cm):
  `sat = SOILLIQ / (WATSAT × DZSOI × 1000)`
- Map to FSAT via sigmoid: `FSAT = fsat_max / (1 + exp(-k × (sat_shallow − sat_crit)))`
- Frozen soils are suppressed automatically (frozen water → SOILICE, not SOILLIQ)
- Ocean cells are masked via NaN in WATSAT

**Comparison baseline:** simple linear precipitation-based FSAT from `MethaneOfflineModel.estimate_fsat_from_precip`.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import importlib
import methane_offline_model
importlib.reload(methane_offline_model)
from methane_offline_model import MethaneOfflineModel

## 1. Load data

In [ ]:
# --- NPP (summed over all PFTs) ---
ds_npp = xr.open_dataset('trace.01-36.22000BP.clm2.NPP.22000BP_decavg_400BCE.nc')
npp_raw = ds_npp.NPP.values.sum(axis=1)          # sum PFTs -> (time, lat, lon)
lat = ds_npp.lat.values
lon = ds_npp.lon.values
time = ds_npp.time.values                         # ka BP
print(f'NPP (all PFTs): {npp_raw.shape}')

# --- Temperature: T10 (10-day mean 2-m temperature) ---
ds_t10 = xr.open_dataset('trace.01-36.22000BP.clm2.T10.22000BP_decavg_400BCE.nc')
temp_raw = ds_t10.T10.values                      # (time, lat, lon), K
print(f'T10: {temp_raw.shape}')

# --- Precipitation ---
ds_p = xr.open_dataset('trace.01-36.22000BP.cam2.PRECT.22000BP_decavg_400BCE.nc')
precip_raw = ds_p.PRECT.values                    # (time, lat, lon), m/s
print(f'PRECT: {precip_raw.shape}')

# --- SOILLIQ (also contains WATSAT and DZSOI) ---
ds_sl = xr.open_dataset('trace.01-36.22000BP.clm2.SOILLIQ.22000BP_decavg_400BCE.nc')
soilliq_raw = ds_sl.SOILLIQ.values               # (time, levsoi, lat, lon), kg/m²
watsat      = ds_sl.WATSAT.values                # (levsoi, lat, lon), mm³/mm³, NaN over ocean
dzsoi       = ds_sl.DZSOI.values                 # (levsoi, lat, lon), m
levsoi      = ds_sl.levsoi.values                # soil layer mid-depths, m
print(f'SOILLIQ: {soilliq_raw.shape}')
print(f'WATSAT:  {watsat.shape}')
print(f'DZSOI:   {dzsoi.shape}')
print(f'Soil layer depths (m): {levsoi.round(3)}')

In [ ]:
# Transpose all 3D fields to (lon, lat, time) to match MethaneOfflineModel convention
npp_model    = np.nan_to_num(np.transpose(npp_raw,    (2, 1, 0)), nan=0.0).clip(0)
temp_model   = np.nan_to_num(np.transpose(temp_raw,   (2, 1, 0)), nan=273.16)
precip_model = np.nan_to_num(np.transpose(precip_raw, (2, 1, 0)), nan=0.0).clip(0)
# SOILLIQ stays (time, levsoi, lat, lon) for the FSAT calculation below

years_bp = np.abs(time) * 1000                    # years before present
years_ce = 1950 - years_bp                        # calendar year (negative = BCE)

model = MethaneOfflineModel(knpp=0.01, q10=1.6, tref=273.16)
area  = model.compute_grid_area(lat, lon)          # (lon, lat), m²
print(f'npp_model:    {npp_model.shape}')
print(f'temp_model:   {temp_model.shape}')
print(f'precip_model: {precip_model.shape}')

## 2. FSAT estimation — precipitation-based baseline

In [ ]:
# Precipitation-based FSAT: linear method from MethaneOfflineModel
# (the same approach used in the original notebook)
fsat_precip = model.estimate_fsat_from_precip(
    precip=precip_model,
    temp=temp_model,
    method='linear'
)                                                  # (lon, lat, time)

print(f'FSAT (precip): shape={fsat_precip.shape}')
land_mask_lonlat = np.isfinite(watsat[0].T)        # (lon, lat) land mask from WATSAT
print(f'  Land mean: {np.nanmean(fsat_precip[land_mask_lonlat]):.4f}')
print(f'  Global mean: {fsat_precip.mean():.4f}')

## 3. FSAT estimation — SOILLIQ-based (new approach)

In [ ]:
def estimate_fsat_from_soilliq(soilliq, watsat, dzsoi,
                                n_layers=4, fsat_max=0.4,
                                sat_crit=0.75, k=10):
    """
    Estimate FSAT from soil liquid water content.

    Parameters
    ----------
    soilliq  : (time, levsoi, lat, lon)  kg/m²
    watsat   : (levsoi, lat, lon)        mm³/mm³, NaN over ocean
    dzsoi    : (levsoi, lat, lon)        m
    n_layers : number of shallow layers to use (default 4, covering 0–12 cm)
    fsat_max : maximum FSAT
    sat_crit : saturation threshold for sigmoid midpoint
    k        : sigmoid steepness

    Returns
    -------
    fsat : (lon, lat, time)   — same convention as MethaneOfflineModel
    """
    # Max water capacity for shallow layers (kg/m²)
    max_water = watsat[:n_layers] * dzsoi[:n_layers] * 1000   # (n_layers, lat, lon)

    # Degree of saturation per layer; sat > 1 indicates ponding
    sat_deg = (soilliq[:, :n_layers, :, :]                    # (time, n_layers, lat, lon)
               / (max_water[np.newaxis] + 1e-10))

    # Depth-weighted mean over shallow layers
    dz = dzsoi[:n_layers]                                     # (n_layers, lat, lon)
    sat_shallow = (np.nansum(sat_deg * dz[np.newaxis], axis=1)
                   / np.nansum(dz[np.newaxis], axis=1))       # (time, lat, lon)

    # Sigmoid mapping: smoothly 0 → fsat_max centred at sat_crit
    fsat_latlon = fsat_max / (1.0 + np.exp(-k * (sat_shallow - sat_crit)))

    # Zero out ocean cells (NaN WATSAT)
    ocean_mask = ~np.isfinite(watsat[0])                      # (lat, lon)
    fsat_latlon[:, ocean_mask] = 0.0

    # Transpose to (lon, lat, time) for MethaneOfflineModel
    fsat = np.transpose(fsat_latlon, (2, 1, 0))               # (lon, lat, time)
    return np.clip(fsat, 0.0, 1.0)

In [ ]:
fsat_soilliq = estimate_fsat_from_soilliq(
    soilliq=soilliq_raw,
    watsat=watsat,
    dzsoi=dzsoi,
    n_layers=4,
    fsat_max=0.4,
    sat_crit=0.75,
    k=10
)                                                              # (lon, lat, time)

print(f'FSAT (SOILLIQ): shape={fsat_soilliq.shape}')
print(f'  Land mean: {np.nanmean(fsat_soilliq[land_mask_lonlat]):.4f}')
print(f'  Global mean: {fsat_soilliq.mean():.4f}')
print(f'  Range: {fsat_soilliq.min():.4f} – {fsat_soilliq.max():.4f}')

# Fraction of land cells with FSAT > 0.05 at any timestep (sanity check)
fsat_timemean_soilliq = fsat_soilliq.mean(axis=2)             # (lon, lat)
wet_frac = (fsat_timemean_soilliq[land_mask_lonlat] > 0.05).mean()
print(f'  Fraction of land cells with time-mean FSAT > 0.05: {wet_frac:.3f}')

## 4. Compute CH4 flux and global emissions — both methods

In [ ]:
# Precipitation-based
f_ch4_precip   = model.compute_methane_flux(npp=npp_model, temp=temp_model,
                                             fsat=fsat_precip)
emissions_precip = model.compute_global_emissions(f_ch4_precip, area)

# SOILLIQ-based
f_ch4_soilliq  = model.compute_methane_flux(npp=npp_model, temp=temp_model,
                                             fsat=fsat_soilliq)
emissions_soilliq = model.compute_global_emissions(f_ch4_soilliq, area)

print('=== Global CH4 emissions ===')
print(f'Precip method:  mean={emissions_precip.mean():.1f}  '
      f'range=[{emissions_precip.min():.1f}, {emissions_precip.max():.1f}] TgCH4/yr')
print(f'SOILLIQ method: mean={emissions_soilliq.mean():.1f}  '
      f'range=[{emissions_soilliq.min():.1f}, {emissions_soilliq.max():.1f}] TgCH4/yr')

## 5. Four-panel comparison plot

In [ ]:
# Pre-compute time means in (lat, lon) for mapping
fsat_mean_precip   = np.mean(fsat_precip,  axis=2).T   # (lat, lon)
fsat_mean_soilliq  = np.mean(fsat_soilliq, axis=2).T   # (lat, lon)

# Shift longitudes 0-360 → -180 to 180 for cartopy
lon_plot = np.where(lon > 180, lon - 360, lon)
sidx = np.argsort(lon_plot)
lon_s = lon_plot[sidx]
fsat_mean_precip_s  = fsat_mean_precip[:, sidx]
fsat_mean_soilliq_s = fsat_mean_soilliq[:, sidx]

# Zonal means
f_ch4_mean_precip  = np.nanmean(f_ch4_precip,  axis=2).T   # (lat, lon)
f_ch4_mean_soilliq = np.nanmean(f_ch4_soilliq, axis=2).T
zonal_precip  = np.nanmean(f_ch4_mean_precip,  axis=1)     # (lat,)
zonal_soilliq = np.nanmean(f_ch4_mean_soilliq, axis=1)

proj = ccrs.Robinson()
pc   = ccrs.PlateCarree()

fig = plt.figure(figsize=(16, 11))
fig.suptitle('SOILLIQ vs Precipitation FSAT: Methane Flux Comparison', fontsize=14, y=1.01)

# ── Panel A: FSAT map – precipitation baseline ──────────────────────────────
ax1 = fig.add_subplot(2, 2, 1, projection=proj)
im1 = ax1.pcolormesh(lon_s, lat, fsat_mean_precip_s,
                     transform=pc, cmap='Blues', vmin=0, vmax=0.4, shading='auto')
ax1.coastlines(linewidth=0.5)
ax1.set_global()
ax1.set_title('(a) Time-mean FSAT — Precipitation method')
plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.04, shrink=0.85,
             label='Wetland fraction')

# ── Panel B: FSAT map – SOILLIQ method ──────────────────────────────────────
ax2 = fig.add_subplot(2, 2, 2, projection=proj)
im2 = ax2.pcolormesh(lon_s, lat, fsat_mean_soilliq_s,
                     transform=pc, cmap='Blues', vmin=0, vmax=0.4, shading='auto')
ax2.coastlines(linewidth=0.5)
ax2.set_global()
ax2.set_title('(b) Time-mean FSAT — SOILLIQ method')
plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.04, shrink=0.85,
             label='Wetland fraction')

# ── Panel C: Global CH4 emissions time series ────────────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(years_ce, emissions_precip,  color='steelblue',  lw=1.5, label='Precip method')
ax3.plot(years_ce, emissions_soilliq, color='darkorange', lw=1.5, label='SOILLIQ method')
ax3.set_xlabel('Calendar year (CE/BCE)')
ax3.set_ylabel('CH4 emissions (TgCH4/yr)')
ax3.set_title('(c) Global CH4 emissions')
ax3.legend()
ax3.grid(True, alpha=0.3)

# ── Panel D: Zonal mean CH4 flux ─────────────────────────────────────────────
ax4 = fig.add_subplot(2, 2, 4)
ax4.plot(lat, zonal_precip,  color='steelblue',  lw=1.5, label='Precip method')
ax4.plot(lat, zonal_soilliq, color='darkorange', lw=1.5, label='SOILLIQ method')
ax4.axhline(0, color='k', lw=0.5)
ax4.set_xlabel('Latitude')
ax4.set_ylabel('Zonal mean CH4 flux (gC/m²/yr)')
ax4.set_title('(d) Zonal mean CH4 flux')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_soilliq_vs_precip.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_soilliq_vs_precip.png')

## 6. Summary statistics

In [ ]:
print('=== FSAT summary (land cells only) ===')
for label, fsat in [('Precip', fsat_precip), ('SOILLIQ', fsat_soilliq)]:
    vals = fsat[land_mask_lonlat, :]   # all land × time values
    print(f'{label:8s}: mean={vals.mean():.4f}  '
          f'p50={np.percentile(vals,50):.4f}  '
          f'p90={np.percentile(vals,90):.4f}  '
          f'max={vals.max():.4f}')

print()
print('=== CH4 emissions summary ===')
for label, em in [('Precip', emissions_precip), ('SOILLIQ', emissions_soilliq)]:
    print(f'{label:8s}: mean={em.mean():.1f}  '
          f'LGM(t=0)={em[0]:.1f}  '
          f'Holocene(t=1600)={em[1600]:.1f}  '
          f'late(t=-1)={em[-1]:.1f} TgCH4/yr')

print()
print('=== Correlation between methods ===')
r_emissions = np.corrcoef(emissions_precip, emissions_soilliq)[0, 1]
r_fsat = np.corrcoef(fsat_precip.ravel(), fsat_soilliq.ravel())[0, 1]
print(f'Pearson r (FSAT, all cells/times): {r_fsat:.4f}')
print(f'Pearson r (global emissions):      {r_emissions:.4f}')

## 7. Time-evolution GIF: CH4 flux across the deglaciation–Holocene

One frame per 100 years (every 10 decadal timesteps), smoothed with a ±100-yr window.
Left panel: SOILLIQ-based CH4 flux spatial map. Right panel: global emissions time series with a moving marker.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter

# --- sampling & smoothing ---
stride    = 10   # 1 frame per 100 years
half      = 10   # ±100-yr smoothing window
t_idx     = np.arange(0, len(time), stride)
years_anim = years_ce[t_idx]

# Sort lons to -180..180 for cartopy
lon_gif  = np.where(lon > 180, lon - 360, lon)
sidx_gif = np.argsort(lon_gif)
lon_sg   = lon_gif[sidx_gif]

# Pre-compute smoothed frames (lon-sorted, lat×lon)
frames_flux = []
for ti in t_idx:
    t0 = max(0, ti - half);  t1 = min(len(time), ti + half + 1)
    fr = np.nanmean(f_ch4_soilliq[:, :, t0:t1], axis=2).T   # (lat, lon)
    frames_flux.append(fr[:, sidx_gif])

vmax = np.nanpercentile(np.concatenate([f.ravel() for f in frames_flux]), 98)

# Smooth global emissions for the time-series panel
kern      = np.ones(20) / 20
em_smooth = np.convolve(emissions_soilliq, kern, mode='same')

# --- build figure ---
proj = ccrs.Robinson();  pc = ccrs.PlateCarree()
fig  = plt.figure(figsize=(14, 5))
gs   = fig.add_gridspec(1, 2, width_ratios=[2, 1], wspace=0.35)
ax_map = fig.add_subplot(gs[0], projection=proj)
ax_ts  = fig.add_subplot(gs[1])

im = ax_map.pcolormesh(lon_sg, lat, frames_flux[0],
                       transform=pc, cmap='YlOrRd',
                       vmin=0, vmax=vmax, shading='auto')
ax_map.coastlines(linewidth=0.5)
ax_map.set_global()
plt.colorbar(im, ax=ax_map, orientation='horizontal',
             pad=0.05, shrink=0.85, label='CH4 flux (gC/m²/yr)')
ttl = ax_map.set_title('')

ax_ts.plot(years_ce, em_smooth, 'k-', lw=1.2)
vline = ax_ts.axvline(years_anim[0], color='red', lw=1.5, ls='--')
ax_ts.set_xlabel('Year (CE/BCE)')
ax_ts.set_ylabel('Global CH4 (TgCH4/yr)')
ax_ts.set_title('Global emissions (SOILLIQ, 200-yr smooth)')
ax_ts.grid(True, alpha=0.3)

def _update(i):
    im.set_array(frames_flux[i].ravel())
    ttl.set_text(f'{int(years_anim[i])} CE/BCE')
    vline.set_xdata([years_anim[i], years_anim[i]])
    return [im, ttl, vline]

ani = FuncAnimation(fig, _update, frames=len(t_idx), interval=80, blit=False)
ani.save('ch4_flux_evolution.gif', writer=PillowWriter(fps=12))
plt.close()
print(f'Saved: ch4_flux_evolution.gif  ({len(t_idx)} frames, 100-yr steps, fps=12)')

In [ ]:
# GIF: time evolution of SOILLIQ-based wetland fraction (FSAT)
frames_fsat = []
for ti in t_idx:
    t0 = max(0, ti - half);  t1 = min(len(time), ti + half + 1)
    fr = np.nanmean(fsat_soilliq[:, :, t0:t1], axis=2).T   # (lat, lon)
    frames_fsat.append(fr[:, sidx_gif])

# NH / SH land-mean FSAT for the time-series panel
_i_30S = np.argmin(np.abs(lat - (-30)))
_i_0   = np.argmin(np.abs(lat -   0 ))
_i_30N = np.argmin(np.abs(lat -  30 ))

def _band_fsat_ts(fsat, i1, i2):
    sub = fsat[:, i1:i2, :]
    lm  = land_mask_lonlat[:, i1:i2]
    return np.where(lm[:, :, np.newaxis], sub, 0.0).sum(axis=(0, 1)) / lm.sum()

kern       = np.ones(20) / 20
fsat_nh_sm = np.convolve(_band_fsat_ts(fsat_soilliq, _i_0,   _i_30N), kern, mode='same')
fsat_sh_sm = np.convolve(_band_fsat_ts(fsat_soilliq, _i_30S, _i_0  ), kern, mode='same')

proj = ccrs.Robinson();  pc = ccrs.PlateCarree()
fig  = plt.figure(figsize=(14, 5))
gs   = fig.add_gridspec(1, 2, width_ratios=[2, 1], wspace=0.35)
ax_map = fig.add_subplot(gs[0], projection=proj)
ax_ts  = fig.add_subplot(gs[1])

im = ax_map.pcolormesh(lon_sg, lat, frames_fsat[0],
                       transform=pc, cmap='Blues',
                       vmin=0, vmax=0.4, shading='auto')
ax_map.coastlines(linewidth=0.5)
ax_map.set_global()
plt.colorbar(im, ax=ax_map, orientation='horizontal',
             pad=0.05, shrink=0.85, label='Wetland fraction (FSAT)')
ttl = ax_map.set_title('')

ax_ts.plot(years_ce, fsat_nh_sm, color='steelblue',  lw=1.5, label='NH tropics (0–30°N)')
ax_ts.plot(years_ce, fsat_sh_sm, color='darkorange', lw=1.5, label='SH tropics (30°S–0°)')
vline2 = ax_ts.axvline(years_anim[0], color='red', lw=1.5, ls='--')
ax_ts.set_xlabel('Year (CE/BCE)')
ax_ts.set_ylabel('FSAT (land mean)')
ax_ts.set_title('Regional wetland fraction (200-yr smooth)')
ax_ts.legend(fontsize=9)
ax_ts.grid(True, alpha=0.3)

def _update_fsat(i):
    im.set_array(frames_fsat[i].ravel())
    ttl.set_text(f'{int(years_anim[i])} CE/BCE')
    vline2.set_xdata([years_anim[i], years_anim[i]])
    return [im, ttl, vline2]

ani2 = FuncAnimation(fig, _update_fsat, frames=len(t_idx), interval=80, blit=False)
ani2.save('fsat_evolution.gif', writer=PillowWriter(fps=12))
plt.close()
print(f'Saved: fsat_evolution.gif  ({len(t_idx)} frames, 100-yr steps, fps=12)')

## 8. Bowl-shape analysis: NH vs SH monsoon dynamics

**Expected orbital story (precession, ~21 kyr cycle):**
- Early Holocene (~11–9 ka BP): NH summer insolation at maximum → strong African & Asian monsoons → wet NH tropics → high CH4
- Mid-Holocene (~6–4 ka BP): declining NH insolation → weakening monsoons → NH drying → CH4 falls to minimum (the "bowl")
- SH: anti-phased; SH summer insolation *increases* through the Holocene → wetting
- Net result in ice cores: CH4 peaks ~9–10 ka BP, reaches a minimum ~5–6 ka BP, slight late-Holocene rise

We test this by:
1. Computing emissions separately for NH tropics (0–30°N), SH tropics (30°S–0°), and boreal (30–70°N)
2. Decomposing into FSAT (orbital/monsoon signal) vs NPP (CO₂-fertilisation trend) components
3. Quantifying Holocene trends and early/mid/late Holocene averages

In [ ]:
# --- Latitude band indices ---
i_30S = np.argmin(np.abs(lat - (-30)))
i_0   = np.argmin(np.abs(lat -   0 ))
i_30N = np.argmin(np.abs(lat -  30 ))
i_70N = np.argmin(np.abs(lat -  70 ))
print(f'30°S → lat[{i_30S}]={lat[i_30S]:.1f}°')
print(f'  0° → lat[{i_0 }]={lat[i_0 ]:.1f}°')
print(f'30°N → lat[{i_30N}]={lat[i_30N]:.1f}°')
print(f'70°N → lat[{i_70N}]={lat[i_70N]:.1f}°')

# --- Regional emissions (SOILLIQ) ---
def band_em(f_ch4, area, i1, i2):
    return model.compute_global_emissions(f_ch4[:, i1:i2, :], area[:, i1:i2])

em_sh   = band_em(f_ch4_soilliq, area, i_30S, i_0  )   # 30°S–0°
em_nh   = band_em(f_ch4_soilliq, area, i_0,   i_30N)   # 0°–30°N
em_bor  = band_em(f_ch4_soilliq, area, i_30N, i_70N)   # 30°–70°N

# --- Regional FSAT land means (removes NPP signal) ---
def band_fsat_ts(fsat, i1, i2):
    sub = fsat[:, i1:i2, :]                             # (lon, lat_sub, time)
    lm  = land_mask_lonlat[:, i1:i2]                    # (lon, lat_sub)
    n   = lm.sum()
    return np.where(lm[:, :, np.newaxis], sub, 0.0).sum(axis=(0, 1)) / n

fsat_sh  = band_fsat_ts(fsat_soilliq, i_30S, i_0  )
fsat_nh  = band_fsat_ts(fsat_soilliq, i_0,   i_30N)
fsat_bor = band_fsat_ts(fsat_soilliq, i_30N, i_70N)

# --- NPP band means (to show CO₂-fertilisation trend) ---
def band_npp_ts(npp, i1, i2):
    sub = npp[:, i1:i2, :]
    lm  = land_mask_lonlat[:, i1:i2]
    n   = lm.sum()
    return np.where(lm[:, :, np.newaxis], sub, 0.0).sum(axis=(0, 1)) / n

npp_nh  = band_npp_ts(npp_model, i_0,   i_30N)
npp_sh  = band_npp_ts(npp_model, i_30S, i_0  )
npp_bor = band_npp_ts(npp_model, i_30N, i_70N)

# --- Simple NH-summer insolation proxy (precession only) ---
# June insolation anomaly ∝ ecc × sin(ω), peaked at ~11 ka BP
T_prec  = 21000    # years
t_peak  = 11000    # years BP when NH summer insolation was maximum
ins_nh_proxy = np.cos(2 * np.pi * (years_bp - t_peak) / T_prec)   # normalised [-1,1]

print('Band emissions computed.')
print(f'  SH tropics mean: {em_sh.mean():.1f} TgCH4/yr')
print(f'  NH tropics mean: {em_nh.mean():.1f} TgCH4/yr')
print(f'  Boreal    mean:  {em_bor.mean():.1f} TgCH4/yr')

In [ ]:
# ── Analysis plot: 6 panels ──────────────────────────────────────────────────
def smooth(ts, w=20):
    return np.convolve(ts, np.ones(w)/w, mode='same')

# Holocene slice (≥ 11.7 ka BP → index ~1030)
h0 = np.argmin(np.abs(years_bp - 11700))
hs = slice(h0, None)
hy = years_ce[hs]

# Holocene period labels for vertical shading
periods = {
    'Early\n(11.7–9 ka)':  (np.argmin(np.abs(years_bp - 11700)),
                             np.argmin(np.abs(years_bp -  9000))),
    'Mid\n(6–4 ka)':       (np.argmin(np.abs(years_bp -  6000)),
                             np.argmin(np.abs(years_bp -  4000))),
    'Late\n(2–0 ka)':      (np.argmin(np.abs(years_bp -  2000)),
                             len(time) - 1),
}
period_colors = ['#d4e6f1', '#d5f5e3', '#fdebd0']

fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=False)
fig.suptitle('Bowl-shape analysis: NH vs SH monsoon dynamics (SOILLIQ FSAT)', fontsize=13)

# ── (a) Regional CH4 emissions ────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(years_ce, smooth(em_nh),  color='steelblue',  lw=1.8, label='NH tropics (0–30°N)')
ax.plot(years_ce, smooth(em_sh),  color='darkorange', lw=1.8, label='SH tropics (30°S–0°)')
ax.plot(years_ce, smooth(em_bor), color='forestgreen',lw=1.8, label='Boreal (30–70°N)', ls='--')
ax.axvline(years_ce[h0], color='gray', lw=1, ls=':', label='Holocene start')
ax.set_ylabel('TgCH4/yr');  ax.set_title('(a) Regional CH4 emissions')
ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

# ── (b) FSAT – orbital/monsoon signal, no NPP ────────────────────────────────
ax = axes[0, 1]
ax.plot(years_ce, smooth(fsat_nh),  color='steelblue',   lw=1.8, label='NH tropics')
ax.plot(years_ce, smooth(fsat_sh),  color='darkorange',  lw=1.8, label='SH tropics')
ax.plot(years_ce, smooth(fsat_bor), color='forestgreen', lw=1.8, label='Boreal', ls='--')
# Overlay scaled insolation proxy
ins_scaled = (ins_nh_proxy - ins_nh_proxy.min()) / (ins_nh_proxy.max() - ins_nh_proxy.min())
ins_scaled = ins_scaled * (fsat_nh.max() - fsat_nh.min()) + fsat_nh.min()
ax.plot(years_ce, ins_scaled, 'k--', lw=1, alpha=0.5, label='NH insolation proxy')
ax.axvline(years_ce[h0], color='gray', lw=1, ls=':')
ax.set_ylabel('FSAT (land mean)');  ax.set_title('(b) Regional FSAT (orbital signal)')
ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

# ── (c) NPP – CO₂-fertilisation trend ────────────────────────────────────────
ax = axes[1, 0]
ax.plot(years_ce, smooth(npp_nh),  color='steelblue',   lw=1.8, label='NH tropics')
ax.plot(years_ce, smooth(npp_sh),  color='darkorange',  lw=1.8, label='SH tropics')
ax.plot(years_ce, smooth(npp_bor), color='forestgreen', lw=1.8, label='Boreal', ls='--')
ax.axvline(years_ce[h0], color='gray', lw=1, ls=':')
ax.set_ylabel('NPP (gC/m²/yr, land mean)');  ax.set_title('(c) Regional NPP (CO₂-fertilisation driver)')
ax.legend(fontsize=8);  ax.grid(True, alpha=0.3)

# ── (d) NH/SH FSAT ratio ─────────────────────────────────────────────────────
ax = axes[1, 1]
ratio = smooth(fsat_nh) / (smooth(fsat_sh) + 1e-10)
ax.plot(years_ce, ratio, color='purple', lw=1.8)
ax.axhline(1.0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.axvline(years_ce[h0], color='gray', lw=1, ls=':')
ax.set_ylabel('NH/SH FSAT ratio')
ax.set_title('(d) NH/SH FSAT ratio\n(>1 = NH wetter; expected to decrease through Holocene)')
ax.grid(True, alpha=0.3)

# ── (e) Holocene zoom: NH FSAT with period shading ───────────────────────────
ax = axes[2, 0]
for (label, (i1, i2)), col in zip(periods.items(), period_colors):
    ax.axvspan(years_ce[i1], years_ce[i2], color=col, alpha=0.6, label=label)
ax.plot(hy, smooth(fsat_nh)[hs],  color='steelblue',  lw=2, label='NH FSAT')
ax.plot(hy, smooth(fsat_sh)[hs],  color='darkorange', lw=2, label='SH FSAT')
ax.plot(hy, ins_scaled[hs],       color='k', lw=1.2, ls='--', alpha=0.6, label='NH ins. proxy')
ax.set_xlabel('Year (CE/BCE)');  ax.set_ylabel('FSAT')
ax.set_title('(e) Holocene FSAT with orbital periods')
ax.legend(fontsize=8, ncol=2);  ax.grid(True, alpha=0.3)

# ── (f) Holocene zoom: global emissions ──────────────────────────────────────
ax = axes[2, 1]
for (label, (i1, i2)), col in zip(periods.items(), period_colors):
    ax.axvspan(years_ce[i1], years_ce[i2], color=col, alpha=0.6, label=label)
ax.plot(hy, smooth(emissions_soilliq)[hs], color='black', lw=2, label='Global (SOILLIQ)')
ax.plot(hy, smooth(em_nh)[hs], color='steelblue',  lw=1.5, ls='--', label='NH tropics')
ax.plot(hy, smooth(em_sh)[hs], color='darkorange', lw=1.5, ls='--', label='SH tropics')
ax.set_xlabel('Year (CE/BCE)');  ax.set_ylabel('TgCH4/yr')
ax.set_title('(f) Holocene CH4 emissions (expected bowl not found?)')
ax.legend(fontsize=8, ncol=2);  ax.grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Year (CE/BCE)')

plt.tight_layout()
plt.savefig('bowl_shape_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bowl_shape_analysis.png')

In [ ]:
# ── Quantitative bowl-shape metrics ──────────────────────────────────────────
from scipy import stats

def holocene_metrics(ts, label):
    """Print trend + early/mid/late Holocene averages for a time series."""
    # Holocene linear trend
    slope, intercept, r, p, se = stats.linregress(hy, ts[hs])
    
    # Period averages
    avgs = {}
    for pname, (i1, i2) in periods.items():
        avgs[pname] = ts[i1:i2].mean()
    
    pnames = list(avgs.keys())
    early, mid, late = avgs[pnames[0]], avgs[pnames[1]], avgs[pnames[2]]
    
    bowl = (early > mid) and (late > mid)     # true bowl shape
    decline = (early > mid) and (mid > late)  # monotonic decline (expected NH)
    
    pattern = 'BOWL ✓' if bowl else ('decline' if decline else 'rise/flat')
    
    print(f'{label}')
    print(f'  Holocene trend: {slope*1000:+.4f} /1000 yr  (p={p:.3f})')
    print(f'  Early={early:.4f}  Mid={mid:.4f}  Late={late:.4f}  → {pattern}')
    return slope, r, early, mid, late

print('=' * 65)
print('QUANTITATIVE BOWL-SHAPE ASSESSMENT (200-yr smoothed)')
print('Expected: NH decline or bowl; SH flat/rise; Global bowl')
print('=' * 65)
print()
print('─── FSAT (pure monsoon/insolation signal) ───')
_, r_nh_f,  *_ = holocene_metrics(smooth(fsat_nh),  'NH tropics FSAT (0–30°N)')
_, r_sh_f,  *_ = holocene_metrics(smooth(fsat_sh),  'SH tropics FSAT (0–30°S)')
_, r_bor_f, *_ = holocene_metrics(smooth(fsat_bor), 'Boreal FSAT    (30–70°N)')
print()
print('─── CH4 emissions (FSAT × NPP × Q10) ───')
holocene_metrics(smooth(em_nh),  'NH tropics emissions')
holocene_metrics(smooth(em_sh),  'SH tropics emissions')
holocene_metrics(smooth(em_bor), 'Boreal emissions')
holocene_metrics(smooth(emissions_soilliq), 'Global emissions')
print()

# NH–SH FSAT anti-phase: expected negative correlation through Holocene
r_anti, p_anti = stats.pearsonr(smooth(fsat_nh)[hs], smooth(fsat_sh)[hs])
print(f'─── NH vs SH FSAT anti-phase test (Holocene only) ───')
print(f'  Pearson r = {r_anti:.3f}  (p={p_anti:.4f})')
print(f'  Negative r expected if NH and SH are anti-phased by orbital forcing')
print()

# NPP trend to quantify masking effect
slope_npp_nh, _, r_npp, p_npp, _ = stats.linregress(hy, smooth(npp_nh)[hs])
slope_fsat_nh, _, _, _, _ = stats.linregress(hy, smooth(fsat_nh)[hs])
print(f'─── Signal masking: NPP vs FSAT Holocene trends (NH tropics) ───')
print(f'  NPP trend:  {slope_npp_nh*1000:+.3f} gC/m²/yr per 1000 yr  '
      f'→ drives emissions UP')
print(f'  FSAT trend: {slope_fsat_nh*1000:+.6f} per 1000 yr  '
      f'→ monsoon signal (expected negative)')
fsat_effect  = slope_fsat_nh / smooth(fsat_nh)[hs].mean()
npp_effect   = slope_npp_nh  / smooth(npp_nh )[hs].mean()
print(f'  Relative rate: NPP={npp_effect*1000:+.3f}%/1000yr, '
      f'FSAT={fsat_effect*1000:+.3f}%/1000yr')
print(f'  → NPP trend dominates FSAT trend by factor: '
      f'{abs(npp_effect/fsat_effect):.1f}x → bowl shape SUPPRESSED')


## 9. Alternative modelling schemes from literature

The original SOILLIQ-based FSAT has two diagnosed problems:
1. **NPP dominance**: CLM2's CO₂-fertilisation drives NPP up ~42% through the Holocene, overpowering the orbital/monsoon FSAT signal (NPP trend 2.1× stronger than FSAT trend in NH tropics).
2. **FSAT doesn't track precession**: SOILLIQ saturation responds to temperature (both hemispheres warm together), not to the anti-phased NH/SH summer insolation signal expected from orbital precession.

Three tractable fixes are implemented below, drawing on the following literature:

### Scheme A — Detrended NPP
**Source**: Kaplan (2002, *Global Biogeochem. Cycles*) showed that substrate limitation from soil carbon pools—not the instantaneous CO₂-fertilised NPP—controls methanogenesis. For the Holocene, the CO₂ rise is a confounding trend unrelated to orbital forcing.  
**Method**: Subtract a 2000-yr Gaussian running mean (σ = 100 timesteps ≈ 1000 yr) from each grid cell's NPP time series, then add back the temporal mean to preserve units. Detrended NPP retains millennial-scale variability (orbital, monsoon) while removing the long-term CO₂-driven rise.

### Scheme B — Seasonal PRECT FSAT
**Source**: Singarayer & Valdes (2011, *Nature*) demonstrated that orbital precession explains virtually all of the Holocene CH4 record through its control on seasonal tropical precipitation: high NH summer (JJA) insolation at 11 ka BP → strong African/Asian monsoon → wet NH tropical wetlands. As NH summer insolation declined, JJA precipitation fell, reducing FSAT and CH4. The anti-phase signal in the SH (DJF) further contributed.  
**Method**: Load JJA and DJF seasonal PRECT files (already available from TraCE). For each grid cell, blend the appropriate seasonal precipitation with the annual mean using latitude-dependent weights (0% seasonal at equator → 100% at ±30°). Apply the exponential FSAT parameterisation from `MethaneOfflineModel`.

### Scheme C — Curvature-TWI SOILLIQ FSAT
**Source**: Gedney & Cox (2003, *GRL*) showed that subgrid topographic variability (Topographic Wetness Index, TWI) controls where water accumulates. Kaplan et al. (2019, *GMD*) validated a slope-based approach for deep-time models.  
**Method**: Since the available ETOPO1 file covers only Antarctica (EPSG:3031), we derive a Topographic Wetland Potential (TWP) from the spatial curvature of long-term mean SOILLIQ saturation. Topographic basins accumulate water → local saturation maxima (negative Laplacian) → high TWP. Slopes drain rapidly → local minima → low TWP. This is conceptually equivalent to a coarse TWI computed directly from the model's own hydrological state.


In [ ]:

# ── Scheme A: Detrended NPP (removes CO₂-fertilisation bias) ─────────────────
# Problem: CLM2's rising CO₂ drives NPP up ~42% through the Holocene, a trend
# unrelated to orbital/monsoon forcing that swamps the FSAT signal.
# Fix: subtract a 2000-yr Gaussian trend (≈ long-term CO₂ response) and add back
# the temporal mean so units remain unchanged. Use fsat_soilliq unchanged.

from scipy.ndimage import gaussian_filter1d

# Gaussian sigma in timesteps: 1000 yr / 10 yr per step = 100 timesteps
# (FWHM ≈ 2350 yr ≈ 2000-yr window as described in plan)
sigma_ts = 100

# npp_model is (lon, lat, time); apply filter along time axis (axis=2)
npp_trend = gaussian_filter1d(npp_model.astype(float), sigma=sigma_ts, axis=2, mode='nearest')
npp_mean  = npp_model.mean(axis=2, keepdims=True)   # (lon, lat, 1) temporal mean

# Detrended NPP: remove slow CO₂ trend, preserve shorter-timescale variability
npp_A = (npp_model - npp_trend + npp_mean).clip(0)  # (lon, lat, time)

print(f'NPP original: mean={npp_model.mean():.4f}  '
      f'Holocene trend: {(npp_model[:,:,-1].mean() - npp_model[:,:,0].mean()):.4f} gC/m²/s')
print(f'NPP trend:    mean={npp_trend.mean():.4f}  '
      f'(should be ~ same as original mean)')
print(f'NPP detrended (A): mean={npp_A.mean():.4f}  '
      f'Holocene trend: {(npp_A[:,:,-1].mean() - npp_A[:,:,0].mean()):.4f} gC/m²/s')
print(f'  (trend should be near zero if detrending successful)')

# Compute CH4 with detrended NPP; keep SOILLIQ-based FSAT unchanged
f_ch4_A = model.compute_methane_flux(npp=npp_A, temp=temp_model, fsat=fsat_soilliq)
em_A    = model.compute_global_emissions(f_ch4_A, area)

print(f'\nScheme A (detrended NPP) emissions: mean={em_A.mean():.1f}  '
      f'range=[{em_A.min():.1f}, {em_A.max():.1f}] TgCH4/yr')


In [ ]:

# ── Scheme B: Seasonal PRECT FSAT (after Singarayer & Valdes 2011 Nature) ───
# NH tropics (0–30°N): JJA (boreal summer monsoon) drives wetland saturation.
# SH tropics (30°S–0°): DJF (austral summer monsoon) drives wetland saturation.
# Blend linearly between seasonal and annual mean as |lat| → 30°.
# This encodes the orbital precession signal into the precipitation FSAT estimate.

ds_jja = xr.open_dataset('trace.01-36.22000BP.cam2.PRECT.22000BP_decavgJJA_400BCE.nc')
ds_djf = xr.open_dataset('trace.01-36.22000BP.cam2.PRECT.22000BP_decavgDJF_400BCE.nc')

# Transpose to (lon, lat, time) — same convention as model
prect_jja = np.nan_to_num(np.transpose(ds_jja.PRECT.values, (2, 1, 0)), nan=0.0).clip(0)
prect_djf = np.nan_to_num(np.transpose(ds_djf.PRECT.values, (2, 1, 0)), nan=0.0).clip(0)
prect_ann = precip_model   # annual mean, already loaded

print(f'JJA PRECT: {prect_jja.shape}')
print(f'DJF PRECT: {prect_djf.shape}')

# ── Latitude-dependent blending ──────────────────────────────────────────────
# w_seasonal: 0 at equator → 1 at ±30° (seasonal signal strengthens with |lat|)
# Then switch sign at equator: JJA for NH, DJF for SH
lat_3d = lat[np.newaxis, :, np.newaxis]   # (1, lat, 1) for broadcasting (lon, lat, time)

# Weight for seasonal component: 0 at equator, 1 at ±30°
w_trop = np.clip(np.abs(lat_3d) / 30.0, 0.0, 1.0)

# Choose seasonal season by hemisphere (JJA for NH, DJF for SH)
prect_seasonal = np.where(lat_3d >= 0, prect_jja, prect_djf)

# Blended: equatorial cells use annual mean; ±30° use purely seasonal
prect_B = (w_trop * prect_seasonal + (1.0 - w_trop) * prect_ann).clip(0)

# Apply the exponential FSAT parameterisation (after Singarayer & Valdes 2011)
fsat_B = model.estimate_fsat_from_precip(
    precip=prect_B,
    temp=temp_model,
    method='exponential'
)

f_ch4_B = model.compute_methane_flux(npp=npp_model, temp=temp_model, fsat=fsat_B)
em_B = model.compute_global_emissions(f_ch4_B, area)

print(f'\nScheme B (seasonal PRECT) emissions: mean={em_B.mean():.1f}  '
      f'range=[{em_B.min():.1f}, {em_B.max():.1f}] TgCH4/yr')
print(f'FSAT_B land mean: {fsat_B[land_mask_lonlat].mean():.4f}')


In [ ]:

# ── Scheme C: Curvature-weighted SOILLIQ FSAT (Gedney & Cox 2003 spirit) ────
# NOTE: The ETOPO1 tif in this repository is in EPSG:3031 (Antarctic polar
# stereographic, 5792×5792 px) and does NOT cover the global land surface.
# Instead we derive the Topographic Wetland Potential (TWP) from the spatial
# curvature of the long-term mean SOILLIQ saturation itself.
#
# Scientific basis: topographic basins accumulate water → SOILLIQ is locally
# high relative to neighbours (a spatial maximum), so its Laplacian is negative.
# Slopes and ridges drain quickly → SOILLIQ is locally low (spatial minimum,
# positive Laplacian). Therefore TWP_proxy = −∇²(sat̄_shallow) captures the
# same hydrological information as a TWI computed from elevation (Gedney & Cox
# 2003; Kaplan et al. 2019 GMD) using only quantities already in the model.

from scipy.ndimage import gaussian_filter, laplace

# Long-term mean shallow saturation (depth-weighted top 4 layers)
n_layers = 4
max_water  = watsat[:n_layers] * dzsoi[:n_layers] * 1000            # (n_layers, lat, lon)
sat_all    = soilliq_raw[:, :n_layers] / (max_water[np.newaxis] + 1e-10)
dz         = dzsoi[:n_layers]                                        # (n_layers, lat, lon)
sat_sh     = (np.nansum(sat_all * dz[np.newaxis], axis=1)
              / np.nansum(dz[np.newaxis], axis=1))                  # (time, lat, lon)
sat_mean   = np.nanmean(sat_sh, axis=0)                             # (lat, lon)

# Zero out ocean (NaN WATSAT)
ocean_mask = ~np.isfinite(watsat[0])
sat_mean[ocean_mask] = 0.0

# Smooth to suppress grid-scale noise, then compute Laplacian
sat_sm   = gaussian_filter(sat_mean, sigma=1.5)                     # (lat, lon)
lap      = laplace(sat_sm)                                          # positive at ridges

# TWP: − Laplacian → positive in basins, negative on slopes; clip and normalise
twp_raw  = np.clip(-lap, 0, None)
twp_norm = twp_raw / (twp_raw.max() + 1e-10)                       # [0,1] (lat, lon)
twp_norm[ocean_mask] = 0.0

print(f'TWP curvature: min={twp_norm.min():.4f}  '
      f'mean(land)={twp_norm[~ocean_mask].mean():.4f}  '
      f'max={twp_norm.max():.4f}')
print(f'Fraction of land cells with TWP > 0.5: '
      f'{(twp_norm[~ocean_mask] > 0.5).mean():.3f}')

def estimate_fsat_twi(soilliq, watsat, dzsoi, twp_latlon,
                      n_layers=4, fsat_max=0.4, sat_crit=0.75, k=10):
    """SOILLIQ FSAT weighted by topographic wetland potential (basin curvature)."""
    max_water_  = watsat[:n_layers] * dzsoi[:n_layers] * 1000
    sat_deg_    = soilliq[:, :n_layers] / (max_water_[np.newaxis] + 1e-10)
    dz_         = dzsoi[:n_layers]
    sat_shallow = (np.nansum(sat_deg_ * dz_[np.newaxis], axis=1)
                   / np.nansum(dz_[np.newaxis], axis=1))            # (time, lat, lon)
    sat_twi     = np.clip(sat_shallow * twp_latlon[np.newaxis], 0, 1)
    fsat_latlont = fsat_max / (1.0 + np.exp(-k * (sat_twi - sat_crit)))
    fsat_latlont[:, ~np.isfinite(watsat[0])] = 0.0
    return np.clip(np.transpose(fsat_latlont, (2, 1, 0)), 0.0, 1.0)  # (lon, lat, time)

fsat_C  = estimate_fsat_twi(soilliq_raw, watsat, dzsoi, twp_norm)
f_ch4_C = model.compute_methane_flux(npp=npp_model, temp=temp_model, fsat=fsat_C)
em_C    = model.compute_global_emissions(f_ch4_C, area)

print(f'\nScheme C (curvature-TWI×SOILLIQ) emissions: mean={em_C.mean():.1f}  '
      f'range=[{em_C.min():.1f}, {em_C.max():.1f}] TgCH4/yr')
print(f'FSAT_C land mean: {fsat_C[land_mask_lonlat].mean():.4f}')


In [ ]:

# ── 4-panel comparison: all four schemes ────────────────────────────────────
def _band_fsat_ts_local(fsat, i1, i2):
    sub = fsat[:, i1:i2, :]
    lm  = land_mask_lonlat[:, i1:i2]
    return np.where(lm[:, :, np.newaxis], sub, 0.0).sum(axis=(0, 1)) / lm.sum()

def smooth20(ts):
    return np.convolve(ts, np.ones(20) / 20, mode='same')

# NH/SH tropical FSAT time series for each scheme
_i_30S = np.argmin(np.abs(lat - (-30)))
_i_0   = np.argmin(np.abs(lat -   0 ))
_i_30N = np.argmin(np.abs(lat -  30 ))

schemes = {
    'Original (SOILLIQ)': (emissions_soilliq, fsat_soilliq, f_ch4_soilliq, 'k',           '-'),
    'A: Detrended NPP':   (em_A,              fsat_soilliq, f_ch4_A,       'steelblue',   '--'),
    'B: Seasonal PRECT':  (em_B,              fsat_B,       f_ch4_B,       'darkorange',  '-.'),
    'C: TWI×SOILLIQ':     (em_C,              fsat_C,       f_ch4_C,       'forestgreen', ':'),
}

# Holocene indices
_h0 = np.argmin(np.abs(years_bp - 11700))
_hs = slice(_h0, None)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Alternative Methane Modelling Schemes: Comparison', fontsize=14)

ax_em  = axes[0, 0]   # (a) global emissions time series
ax_hol = axes[0, 1]   # (b) Holocene zoom
ax_rat = axes[1, 0]   # (c) NH/SH FSAT ratio
ax_zon = axes[1, 1]   # (d) zonal mean CH4 flux

for label, (em, fsat, f_ch4, col, ls) in schemes.items():
    # (a) Global emissions
    ax_em.plot(years_ce, smooth20(em), color=col, ls=ls, lw=1.8, label=label)

    # (b) Holocene zoom
    ax_hol.plot(years_ce[_hs], smooth20(em)[_hs], color=col, ls=ls, lw=1.8, label=label)

    # (c) NH/SH FSAT ratio
    fsat_nh_ts = _band_fsat_ts_local(fsat, _i_0, _i_30N)
    fsat_sh_ts = _band_fsat_ts_local(fsat, _i_30S, _i_0)
    ratio = smooth20(fsat_nh_ts) / (smooth20(fsat_sh_ts) + 1e-10)
    ax_rat.plot(years_ce, ratio, color=col, ls=ls, lw=1.8, label=label)

    # (d) Zonal mean CH4 flux (time mean)
    zonal = np.nanmean(f_ch4, axis=(0, 2))   # mean over lon and time → (lat,)
    ax_zon.plot(lat, zonal, color=col, ls=ls, lw=1.8, label=label)

# (a)
ax_em.set_xlabel('Year (CE/BCE)'); ax_em.set_ylabel('TgCH4/yr')
ax_em.set_title('(a) Global CH4 emissions (200-yr smooth)')
ax_em.legend(fontsize=9); ax_em.grid(True, alpha=0.3)
ax_em.axvline(years_ce[_h0], color='gray', lw=1, ls=':', label='Holocene start')

# (b)
ax_hol.set_xlabel('Year (CE/BCE)'); ax_hol.set_ylabel('TgCH4/yr')
ax_hol.set_title('(b) Holocene zoom — does any scheme show the bowl?')
for label_p, (i1, i2), col_p in zip(
        ['Early\n(9–11 ka)', 'Mid\n(4–6 ka)', 'Late\n(0–2 ka)'],
        [(np.argmin(np.abs(years_bp-11000)), np.argmin(np.abs(years_bp-9000))),
         (np.argmin(np.abs(years_bp-6000)),  np.argmin(np.abs(years_bp-4000))),
         (np.argmin(np.abs(years_bp-2000)),  len(time)-1)],
        ['#d4e6f1', '#d5f5e3', '#fdebd0']):
    ax_hol.axvspan(years_ce[i1], years_ce[i2], color=col_p, alpha=0.5)
ax_hol.legend(fontsize=9); ax_hol.grid(True, alpha=0.3)

# (c)
ax_rat.axhline(1.0, color='k', lw=0.8, ls='--', alpha=0.5)
ax_rat.set_xlabel('Year (CE/BCE)'); ax_rat.set_ylabel('NH FSAT / SH FSAT')
ax_rat.set_title('(c) NH/SH tropical FSAT ratio\n(expected to fall through Holocene)')
ax_rat.legend(fontsize=9); ax_rat.grid(True, alpha=0.3)
ax_rat.axvline(years_ce[_h0], color='gray', lw=1, ls=':')

# (d)
ax_zon.set_xlabel('Latitude'); ax_zon.set_ylabel('CH4 flux (gC/m²/yr)')
ax_zon.set_title('(d) Time-mean zonal CH4 flux')
ax_zon.axhline(0, color='k', lw=0.5); ax_zon.axvline(0, color='gray', lw=0.5)
ax_zon.legend(fontsize=9); ax_zon.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('alternative_schemes_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: alternative_schemes_comparison.png')
